In [6]:
import boto3
import json
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("silver_brasileirao") \
    .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.2.0,org.apache.hadoop:hadoop-aws:3.3.4") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://cvmc-minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()

s3 = boto3.client(
    's3',
    endpoint_url='http://cvmc-minio:9000',
    aws_access_key_id='minioadmin',
    aws_secret_access_key='minioadmin'
)

print(f"Spark version: {spark.version}")
print("Delta + MinIO configurados!")

Spark version: 3.5.0
Delta + MinIO configurados!


In [7]:
# =====================
# PARÂMETROS DE EXECUÇÃO
# =====================
LEAGUE = "brasileirao-serie-a"
SEASON = "2026"
ENVIRONMENT = "dev"
REPROC_MODE = False

# =====================
# CONFIGURAÇÕES FIXAS
# =====================
BUCKET = "eng-prd-data-lake"
RAW_PATH = f"s3a://{BUCKET}/{ENVIRONMENT}/raw/{LEAGUE}"
SILVER_PATH = f"s3a://{BUCKET}/{ENVIRONMENT}/silver/{LEAGUE}"

DEDUP_KEYS = {
    "standing": {
        "pk": ["season", "record.teamId"],
        "order_col": "ingested_at"
    },
    "rounds": {
        "pk": ["record.id"],
        "order_col": "ingested_at"
    },
    "events": {
        "pk": ["record.events.id"],
        "order_col": "ingested_at"
    },
    "players": {
        "pk": ["record.players.id", "record.teams.teamId", "season"],
        "order_col": "ingested_at"
    },
    "team_stats": {
        "pk": ["record.team.teamId", "season"],
        "order_col": "ingested_at"
    },
    "player_stats": {
    "pk": ["record.team.teamId", "season"],
    "order_col": "ingested_at"
    }
}

print(f"League: {LEAGUE} | Season: {SEASON} | Env: {ENVIRONMENT} | Reproc: {REPROC_MODE}")
print(f"Raw:    {RAW_PATH}")
print(f"Silver: {SILVER_PATH}")

League: brasileirao-serie-a | Season: 2026 | Env: dev | Reproc: False
Raw:    s3a://eng-prd-data-lake/dev/raw/brasileirao-serie-a
Silver: s3a://eng-prd-data-lake/dev/silver/brasileirao-serie-a


In [8]:
from pyspark.sql.functions import col, row_number
from pyspark.sql.window import Window

def raw_to_silver(data_type):
    raw = f"{RAW_PATH}/{data_type}"
    silver = f"{SILVER_PATH}/{data_type}"
    config = DEDUP_KEYS[data_type]
    pks = config["pk"]
    order_col = config["order_col"]

    print(f"Lendo {data_type} da Raw...")
    
    df = spark.read.format("delta").load(raw)
    print(raw)
    # Deduplicação — mantém o registro mais recente por chave
    window = Window.partitionBy([col(pk) for pk in pks]) \
                   .orderBy(col(order_col).desc())

    df_dedup = df.withColumn("_rank", row_number().over(window)) \
                 .filter(col("_rank") == 1) \
                 .drop("_rank")

    print(f"Registros após dedup: {df_dedup.count()}")

    df_dedup.write.format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .partitionBy("season") \
        .save(silver)

    print(f"[OK] {data_type} salvo na Silver!\n")

for data_type in ["standing", "rounds", "events", "players", "team_stats", "player_stats"]:
    raw_to_silver(data_type)

print("Camada Silver finalizada!")

Lendo standing da Raw...
s3a://eng-prd-data-lake/dev/raw/brasileirao-serie-a/standing
Registros após dedup: 20
[OK] standing salvo na Silver!

Lendo rounds da Raw...
s3a://eng-prd-data-lake/dev/raw/brasileirao-serie-a/rounds
Registros após dedup: 7
[OK] rounds salvo na Silver!

Lendo events da Raw...
s3a://eng-prd-data-lake/dev/raw/brasileirao-serie-a/events
Registros após dedup: 10
[OK] events salvo na Silver!

Lendo players da Raw...
s3a://eng-prd-data-lake/dev/raw/brasileirao-serie-a/players
Registros após dedup: 327
[OK] players salvo na Silver!

Lendo team_stats da Raw...
s3a://eng-prd-data-lake/dev/raw/brasileirao-serie-a/team_stats
Registros após dedup: 1
[OK] team_stats salvo na Silver!

Lendo player_stats da Raw...
s3a://eng-prd-data-lake/dev/raw/brasileirao-serie-a/player_stats
Registros após dedup: 1
[OK] player_stats salvo na Silver!

Camada Silver finalizada!


In [10]:
from pyspark.sql.functions import explode, col

df = spark.read.format("delta").load(f"{SILVER_PATH}/player_stats")

df_exploded = df.select(
    col("record.team.teamName").alias("team_name"),
    explode(col("record.player_stats.both")).alias("player")
).select(
    "team_name",
    col("player.id").alias("player_id"),
    col("player.name").alias("player_name"),
    col("player.position").alias("position"),
    explode(col("player.stats")).alias("stat")
).select(
    "team_name", "player_id", "player_name", "position",
    col("stat.eventId"),
    col("stat.goals"),
    col("stat.goalAssist"),
    col("stat.minutesPlayed"),
    col("stat.rating"),
    col("stat.expectedGoals"),
    col("stat.shots")
)

df_exploded.show(10, truncate=False)

+-----------+---------+----------------+--------+--------+-----+----------+-------------+------+-------------+-----+
|team_name  |player_id|player_name     |position|eventId |goals|goalAssist|minutesPlayed|rating|expectedGoals|shots|
+-----------+---------+----------------+--------+--------+-----+----------+-------------+------+-------------+-----+
|Corinthians|115182   |André Carrillo  |M       |15237887|0    |0         |63           |7.10  |0.0000       |0    |
|Corinthians|115182   |André Carrillo  |M       |15237907|0    |0         |69           |6.90  |0.0102       |1    |
|Corinthians|115182   |André Carrillo  |M       |15237919|0    |0         |90           |7.10  |0.0952       |1    |
|Corinthians|115182   |André Carrillo  |M       |15369877|0    |0         |8            |6.70  |0.0000       |0    |
|Corinthians|124737   |Gabriel Paulista|D       |15237887|0    |0         |90           |6.50  |0.0000       |0    |
|Corinthians|124737   |Gabriel Paulista|D       |15237907|1    |